# Implement Lakeflow Jobs with Azure Databricks

> **Industry context - NexaTel Communications**
>
> NexaTel is a mid-size European telecom operator serving 5 million mobile subscribers.
> Every hour the network switching infrastructure generates **Call Detail Records (CDR)**
> that must be processed, cleansed, and aggregated before the monthly billing run.
> The data engineering team orchestrates this pipeline as a **Lakeflow Job** with four
> notebook tasks forming a DAG:
>
> | Task | Notebook | Depends on | Run-if |
> |---|---|---|---|
> | `ingest_cdr` | `01_ingest_cdr` | *(entry point)* | - |
> | `cleanse_cdr` | `02_cleanse_cdr` | `ingest_cdr` | All succeeded |
> | `aggregate_usage` | `03_aggregate_usage` | `cleanse_cdr` | All succeeded |
> | `error_handler` | `04_error_handler` | `cleanse_cdr` | At least one failed |
>
> All demos in this notebook are grounded in this telecommunications scenario.
> Notebook task files are in `Demo/pipelines/11-lakeflow-jobs/`.


In [ ]:
from scripts.setup import setup_11

spark.sql("USE CATALOG trainer_demo")
setup_11(spark)

## Create job setup and configuration

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.create-lakeflow-job.png)

### Demo: Create a Lakeflow Job for the NexaTel CDR Pipeline

**Trainer instructions - create the job in the Databricks UI:**

1. In the sidebar, go to **Jobs & Pipelines** -> **Create** -> **Job**.
2. Name the job: `NexaTel CDR Pipeline`.
3. **Task 1 - `ingest_cdr`**
   - Type: **Notebook**
   - Source: **Workspace** -> navigate to `pipelines/11-lakeflow-jobs/01_ingest_cdr`
   - Compute: **Serverless**
4. **Task 2 - `cleanse_cdr`**
   - Type: **Notebook** -> `02_cleanse_cdr`
   - Depends on: `ingest_cdr` | Run-if: **All succeeded**
5. **Task 3 - `aggregate_usage`**
   - Type: **Notebook** -> `03_aggregate_usage`
   - Depends on: `cleanse_cdr` | Run-if: **All succeeded**
6. **Task 4 - `error_handler`**
   - Type: **Notebook** -> `04_error_handler`
   - Depends on: `cleanse_cdr` | Run-if: **At least one failed**
7. Add job-level **Parameters**: `source_catalog=trainer_demo`, `source_schema=demo_11`, `billing_month=2024-12`

**Trainer talking points:**
- The DAG editor shows the diamond shape: `cleanse_cdr` has two downstream paths.
- Tasks 3 and 4 are **mutually exclusive**: only one will run per job run.
- Serverless compute: no cluster warm-up, tasks start immediately.
- All four tasks share the same job-level parameters (key-value pairs passed to all tasks).
- Show the **Task source** options: Workspace vs Git provider for version-controlled notebooks.


In [ ]:
# -- Preview raw CDR data: what the job will process --------------------------
# Trainer: show the raw data the job ingests.
# Highlight the intentional quality issues that cleanse_cdr must handle.

from pyspark.sql import functions as F

raw_cdr = spark.read.table("trainer_demo.demo_11.raw_cdr")

print(f"Total CDR records: {raw_cdr.count():,}")
print("\nSchema:")
raw_cdr.printSchema()

print("\nSample records (5 rows):")
raw_cdr.select(
    "cdr_id", "subscriber_id", "call_type",
    "duration_seconds", "termination_code", "roaming"
).show(5, truncate=False)

# Show call type distribution
print("\nCall type distribution:")
raw_cdr.groupBy("call_type").count().orderBy("call_type").show()

# Intentional quality issues
print("=== Quality issues the pipeline must handle ===")
print(f"  NULL subscriber_id    : {raw_cdr.filter(F.col('subscriber_id').isNull()).count():>6,}")
print(f"  Invalid subscriber_id : {raw_cdr.filter(F.col('subscriber_id').isNotNull() & ~F.col('subscriber_id').rlike(r'^SUB-')).count():>6,}")
print(f"  Negative duration     : {raw_cdr.filter(F.col('duration_seconds') < 0).count():>6,}")
print(f"  Zero duration         : {raw_cdr.filter(F.col('duration_seconds') == 0).count():>6,}")


In [ ]:
# -- Demonstrate task dependencies and run-if conditions ----------------------
# Trainer: use this to explain WHEN each run-if condition applies.
# Map the six conditions to realistic NexaTel pipeline scenarios.

from pyspark.sql import functions as F

raw_cdr = spark.read.table("trainer_demo.demo_11.raw_cdr")
total        = raw_cdr.count()
null_subs    = raw_cdr.filter(F.col("subscriber_id").isNull()).count()
bad_duration = raw_cdr.filter(F.col("duration_seconds") <= 0).count()
failed_calls = raw_cdr.filter(F.col("termination_code").isin("FAILED","DROPPED")).count()

print("=== NexaTel CDR quality profile ===")
print(f"  Total raw records        : {total:>8,}")
print(f"  NULL subscriber_id       : {null_subs:>8,}  -> cleanse_cdr will reject these")
print(f"  Invalid duration (<= 0)  : {bad_duration:>8,}  -> cleanse_cdr will reject these")
print(f"  Failed/dropped calls     : {failed_calls:>8,}  -> kept in silver; used in KPIs")
print()

# Illustrate the six run-if conditions with telecom examples
conditions = [
    ("All succeeded",          "aggregate_usage after cleanse_cdr -- only bill clean data"),
    ("At least one succeeded", "send partial billing report even if one branch fails"),
    ("None failed",            "send SLA notification regardless of skips"),
    ("All done",               "clean up temp tables after any pipeline outcome"),
    ("At least one failed",    "error_handler fires when cleanse_cdr raises an exception"),
    ("All failed",             "emergency rollback when ALL upstream tasks failed"),
]
print("=== Run-if condition reference ===")
for cond, example in conditions:
    print(f"  {cond:<28} -> {example}")

# Demonstrate dbutils.jobs.taskValues concept
print("""
=== Task values: passing data between tasks ===
In a real Lakeflow Job, Task 1 sets:
  dbutils.jobs.taskValues.set(key="total_raw_records", value=50000)

Task 2 reads it:
  total = dbutils.jobs.taskValues.get(taskKey="ingest_cdr", key="total_raw_records")

This avoids writing coordination data to a table and keeps the job self-contained.
""")


## Configure job triggers

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.configure-job-triggers.png)

### Demo: Configure Job Triggers for the NexaTel CDR Pipeline

**Scenario:** NexaTel wants the CDR pipeline to start automatically based on:

| Trigger type | NexaTel use case |
|---|---|
| **File arrival** | New hourly CDR batch files land in a Unity Catalog volume |
| **Table update** | An upstream network-events job updates `raw_cdr` table |
| **Continuous** | A near-real-time fraud detection job must never pause |

**Trainer talking points - File arrival trigger:**
- **Jobs & Pipelines** -> `NexaTel CDR Pipeline` -> **Job details** -> **Add trigger** -> **File arrival**
- Path: `/Volumes/trainer_demo/demo_11/cdr_landing/`
- Only NEW files trigger a run - overwriting the same filename does NOT re-trigger.
- Enable **file events** on the external location for production (removes the 50-job limit and 10,000-file cap).
- Advanced: set **Wait after last change** to 300 seconds to batch rapid file drops.

**Trainer talking points - Table update trigger:**
- Trigger type: **Table update**
- Monitor table: `trainer_demo.demo_11.raw_cdr`
- Choose **Any table is updated** or **All tables are updated** (if monitoring multiple source tables).
- Show dynamic parameter: `{{job.trigger.table_update.updated_tables}}`
- Optional: `{{job.trigger.table_update.trainer_demo.demo_11.raw_cdr.version}}` to track Delta version.

**Trainer talking points - Continuous trigger:**
- Trigger type: **Continuous**, Task retry mode: **On failure**
- Used for the fraud detection streaming job that must process CDRs in near real-time.
- Cost implication: job runs 24/7 - use serverless compute to optimize cost.
- Exponential backoff is automatically applied on repeated failures (shown in the restarts section).


In [ ]:
# -- File arrival trigger: demonstrate the monitored path and landing zone -----
# Trainer: show the CDR landing volume path that would be configured in the
# trigger dialog. Then simulate a file arriving by writing sample data.

from pyspark.sql import functions as F

# Show existing volumes in demo_11 schema
print("Volumes in trainer_demo.demo_11:")
try:
    spark.sql("SHOW VOLUMES IN trainer_demo.demo_11").show(truncate=False)
except Exception as e:
    print(f"  (no volumes found - create one for the demo): {e}")

# Illustrate: a file arrival trigger would watch this path
trigger_path = "/Volumes/trainer_demo/demo_11/cdr_landing/"
print(f"\nFile arrival trigger would monitor: {trigger_path}")

# Show how a new CDR file would look when it lands
print("\nSimulating a new CDR file landing in the volume...")
sample_cdr = (
    spark.read.table("trainer_demo.demo_11.raw_cdr")
    .limit(100)
    .select("cdr_id", "subscriber_id", "call_type", "duration_seconds", "call_start_ts")
)
print(f"Sample file: {sample_cdr.count()} CDR records ready to be written to the landing volume")
sample_cdr.show(3)

print("""
In a production setup:
  1. Network switches write CDR files to /Volumes/trainer_demo/demo_11/cdr_landing/
  2. File arrival trigger detects the new file (best-effort check every ~1 minute)
  3. Trigger fires the NexaTel CDR Pipeline job automatically
  4. ingest_cdr task reads the new file and validates it

Important: only one job run starts per trigger check window.
           Use 'Wait after last change' to batch multiple files into one run.
""")


In [ ]:
# -- Table update trigger: show monitored table history and dynamic params -----
# Trainer: show the Delta table history to illustrate what commit events
# the table_update trigger would respond to. Show dynamic parameter syntax.

from pyspark.sql import functions as F

# Show Delta history for raw_cdr table
print("Delta table history for trainer_demo.demo_11.raw_cdr:")
spark.sql("""
    DESCRIBE HISTORY trainer_demo.demo_11.raw_cdr
    LIMIT 5
""").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

# Show what the trigger would do for each commit
print("""
=== Table update trigger behaviour for raw_cdr ===

Each Delta commit (write, merge, delete) to raw_cdr creates a new version.
The trigger fires after detecting a new version.

Dynamic parameters available in the triggered job run:

  {{job.trigger.table_update.updated_tables}}
      Value: ["trainer_demo.demo_11.raw_cdr"]

  {{job.trigger.table_update.trainer_demo.demo_11.raw_cdr.version}}
      Value: 3  (the Delta version that triggered this run)

  {{job.trigger.table_update.trainer_demo.demo_11.raw_cdr.commit_timestamp.iso_datetime}}
      Value: 2024-12-01T02:00:00Z

Advanced timing options:
  Minimum time between triggers: 3600 seconds (hourly batch constraint)
  Wait after last change:        300 seconds  (batch rapid writes)
""")

# Show current table version and row count for context
current = spark.read.table("trainer_demo.demo_11.raw_cdr")
print(f"Current raw_cdr: {current.count():,} records")
print("Termination code breakdown (shows the variety of events the trigger would process):")
current.groupBy("termination_code").count().orderBy("termination_code").show()


## Schedule a job

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.schedule-job.png)

### Demo: Schedule the NexaTel CDR Pipeline

**Scenario:** NexaTel billing requires three recurring schedules:
- **Nightly at 02:00 UTC** - process previous day's CDR (regulatory deadline)
- **First of each month at 03:00 UTC** - generate monthly invoice aggregates
- **Weekdays at 05:00 UTC (= 06:00 CET)** - daily KPI report for the operations centre

**Trainer instructions:**
1. Open `NexaTel CDR Pipeline` -> **Job details** -> **Add trigger** -> **Scheduled**
2. Demonstrate **Simple schedule**: set **12 hours** interval - point out start time is chosen by scheduler.
3. Switch to **Advanced schedule** -> select **Show Cron Syntax**.
4. Set **Time zone**: UTC (explain why UTC avoids DST issues for European operators).

**Cron expressions for NexaTel:**

| Schedule | Cron expression | Field order: SS MM HH DOM MON DOW |
|---|---|---|
| Every night at 02:00 UTC | `0 0 2 * * ?` | DOM/DOW = `?` when the other is `*` |
| 1st of month at 03:00 UTC | `0 0 3 1 * ?` | DOM = 1, DOW = ? |
| Weekdays at 05:00 UTC | `0 0 5 ? * MON-FRI` | DOW = MON-FRI, DOM = ? |
| Every 4 hours | `0 0 */4 * * ?` | `/` increment operator |
| Last Friday of month 06:30 UTC | `0 30 6 ? * 6L` | `L` = last occurrence |

**Talking points:**
- Use UTC for jobs serving European customers to avoid DST ambiguity.
- **Maximum concurrent runs = 1** for the billing job - overlapping runs would double-bill customers.
- Minimum interval enforced by Databricks: 10 seconds regardless of configuration.
- `?` in day-of-month and day-of-week is required when specifying only one of the two.


In [ ]:
# -- Cron expression examples for NexaTel telecom scheduling scenarios --------
# Trainer: run this cell to display cron expressions with explanations.
# Walk through each field and use the NexaTel examples to make it concrete.

from datetime import datetime, timezone

# Reference table of cron fields
print("=== Cron expression field reference ===")
print(f"  {'Field':<15} {'Required':<10} {'Allowed values':<20} {'Special characters'}")
print(f"  {'-'*15} {'-'*10} {'-'*20} {'-'*25}")
fields = [
    ("Seconds",      "Yes", "0-59",       ", - * /"),
    ("Minutes",      "Yes", "0-59",       ", - * /"),
    ("Hours",        "Yes", "0-23",       ", - * /"),
    ("Day of month", "Yes", "1-31",       ", - * ? / L W"),
    ("Month",        "Yes", "1-12 or JAN-DEC", ", - * /"),
    ("Day of week",  "Yes", "1-7 or SUN-SAT",  ", - * ? / L #"),
    ("Year",         "No",  "1970-2099",  ", - * /"),
]
for name, req, vals, chars in fields:
    print(f"  {name:<15} {req:<10} {vals:<20} {chars}")

# NexaTel schedule examples with explanations
print("\n=== NexaTel CDR Pipeline - schedule examples ===")
schedules = [
    ("0 0 2 * * ?",        "Nightly CDR processing",           "Every day at 02:00 UTC"),
    ("0 0 3 1 * ?",        "Monthly billing run",              "1st of each month at 03:00 UTC"),
    ("0 0 5 ? * MON-FRI",  "Weekday operations KPI report",    "Mon-Fri at 05:00 UTC (= 06:00 CET)"),
    ("0 0 */4 * * ?",      "4-hourly incremental CDR load",    "Every 4 hours starting at midnight"),
    ("0 30 6 ? * 6L",      "Monthly SLA report",               "Last Friday of the month at 06:30 UTC"),
    ("0 0 8 ? * 6#3",      "Third-Friday fraud review",        "3rd Friday of the month at 08:00 UTC"),
]
print(f"\n  {'Cron expression':<26} {'Label':<40} {'Plain English'}")
print(f"  {'-'*26} {'-'*40} {'-'*45}")
for expr, label, desc in schedules:
    print(f"  {expr:<26} {label:<40} {desc}")

print("""
Special character guide:
  *  = every value       (every hour, every day)
  ?  = no specific value (use in DOM or DOW when the other is specified)
  -  = range             (MON-FRI  = weekdays)
  ,  = list              (MON,WED,FRI = three days)
  /  = increment         (*/4 in hours = every 4 hours)
  L  = last              (6L = last Friday of month)
  #  = nth occurrence    (6#3 = 3rd Friday of month)

TIP: maximum concurrent runs = 1 prevents the nightly billing job from
     overlapping if it runs longer than 24 hours during a data incident.
""")


## Configure job alerts

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.configure-job-alerts.png)

### Demo: Configure Job Alerts for the NexaTel CDR Pipeline

**Scenario:** NexaTel's on-call data engineering team needs immediate notification when:
- The nightly CDR pipeline **fails** (billing data may be incomplete - SLA impact)
- A run takes **> 45 minutes** (SLA breach - normal runtime is ~20 minutes)
- The pipeline **starts** (regulatory audit log requirement)

**Trainer instructions - configure job-level notifications:**
1. Open `NexaTel CDR Pipeline` -> **Job details** -> **Job notifications** -> **Edit notifications**
2. Add **Email** destination -> events: **Failure** and **Duration warning** (threshold: 45 min)
3. Add **Start** event to a dedicated audit email alias
4. Add a **Slack** or **Microsoft Teams** destination (workspace admin creates these in Admin Settings -> Notifications)

**Trainer instructions - configure task-level notifications:**
1. Edit `cleanse_cdr` task -> **Notifications** -> **Add**
2. Add email -> event: **Failure**
3. Enable **Mute notifications until the last retry** (prevents 4 alerts for 3 retries + final failure)

**Alert design for NexaTel:**

| Event | Destination | Rationale |
|---|---|---|
| Job start | Audit email alias | Regulatory compliance log |
| Job failure | PagerDuty (webhook) | Immediate on-call escalation |
| Task failure (cleanse_cdr) | Teams channel | Dev team visibility into which step failed |
| Duration warning > 45 min | Engineering email | Capacity planning investigation |
| Streaming backlog | PagerDuty | Real-time fraud job health monitor |

**Talking point - alert fatigue:**
With 3 retry attempts, mute task notifications until last retry.
A single transient network blip generates 4 alerts without this setting.
Use duration warnings based on historical P95 runtime, not P50 - set threshold at P95 + 50%.


In [ ]:
# -- Webhook payload example for NexaTel job failure notification -------------
# Trainer: show what JSON payload Databricks sends to an HTTP webhook.
# NexaTel's internal incident system receives this and creates a ticket.

import json

# Simulated webhook payload sent by Databricks on job failure
sample_payload = {
    "event_type": "jobs.on_failure",
    "workspace_id": "1234567890123456",
    "run": {
        "job_id": 42,
        "run_id": 98765,
        "run_name": "NexaTel CDR Pipeline",
        "run_page_url": "https://adb-1234567890.azuredatabricks.net/jobs/42/runs/98765",
        "state": {
            "life_cycle_state": "TERMINATED",
            "result_state": "FAILED",
            "state_message": "Task cleanse_cdr failed: NullPointerException at line 47"
        },
        "start_time": 1714636800000,
        "end_time":   1714639920000,
        "trigger": "PERIODIC"
    }
}

print("=== Sample webhook payload (jobs.on_failure) ===")
print(json.dumps(sample_payload, indent=2))

# Compute duration from payload timestamps
duration_s = (sample_payload["run"]["end_time"] - sample_payload["run"]["start_time"]) / 1000
print(f"\nComputed run duration: {duration_s:.0f} seconds ({duration_s/60:.1f} minutes)")

print("""
Key fields for NexaTel's incident management system:
  event_type     -> route to correct alert queue (failure vs warning vs backlog)
  run_page_url   -> one-click link to the job run for the on-call engineer
  state_message  -> root cause text for the PagerDuty incident description
  start_time /
  end_time       -> compute duration to detect SLA breach automatically

How to use it:
  1. In Admin Settings -> Notifications -> add webhook destination with your endpoint URL
  2. The endpoint receives POST requests with this JSON body on each event
  3. Parse event_type to route: on_failure -> PagerDuty, on_start -> audit log
  4. Use Azure Logic Apps or Power Automate to convert webhooks into ITSM tickets

Tip: configure up to 3 system destinations per event type per job.
""")


## Configure automatic restarts

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.configure-job-automatic-restarts.png)

### Demo: Configure Automatic Restarts for the NexaTel CDR Pipeline

**Scenario:** NexaTel's CDR pipeline processes millions of records daily.
Common transient failures in a telecom environment:
- Network timeout reading from Azure data lake during peak hours
- Transient Delta table lock contention when two jobs write concurrently
- Serverless pool capacity constraint causing cold-start delay

**Trainer instructions - task-level retry policy:**
1. Edit `cleanse_cdr` task -> **+ Add** next to **Retries**
2. Max retries: **3** | Retry interval: **60000 ms** (1 minute)
3. Explanation: interval = ms from START of failed run to START of retry (not from the failure point)
4. Example: task runs 30s then fails -> retry starts 30s after the failure (60s - 30s elapsed)

**Trainer instructions - timeout thresholds:**
1. Edit `aggregate_usage` task -> **Metric thresholds**
2. Metric: **Run duration** | Warning: **30 minutes** | Timeout: **60 minutes**
3. A timeout terminates the task and counts as one retry attempt
4. Pattern: normal runtime 20 min -> warning at 30 min -> timeout at 60 min

**Trainer instructions - continuous job retry (fraud detection scenario):**
1. Create a new job -> **Add trigger** -> **Continuous**, Task retry mode: **On failure**
2. Show the exponential backoff algorithm (see code cell below)
3. Monitor continuous jobs via **Job details** panel: shows consecutive failures + next retry time

**Production best practices for NexaTel:**

| Task | Retries | Interval | Rationale |
|---|---|---|---|
| `ingest_cdr` | 3 | 30 s | Transient file-read timeout |
| `cleanse_cdr` | 3 | 60 s | Lock contention resolves quickly |
| `aggregate_usage` | 2 | 120 s | Heavy compute - allow recovery time |
| `error_handler` | 0 | - | Must not retry: append-only logging |

**Talking point - idempotency:**
- `cleanse_cdr` uses OVERWRITE mode -> **idempotent** (safe to retry multiple times).
- `error_handler` uses APPEND mode -> **NOT idempotent** (retrying creates duplicate log entries).
- Non-idempotent tasks (invoice sends, external API calls) should disable auto-optimization on serverless.


In [ ]:
# -- Simulate task retry behaviour for the cleanse_cdr step ------------------
# Trainer: this cell SIMULATES the retry logic configured on cleanse_cdr.
# Walk through the retry configuration dialog in the UI while this runs.
# In the real Lakeflow Job, the retry is managed by the platform automatically.

import random
import time

def run_cleanse_cdr(attempt: int) -> dict:
    """Simulate cleanse_cdr with decreasing failure probability per retry."""
    # Simulate transient failure: 70% chance on attempt 1, decreasing each time
    fail_probability = max(0.0, 0.70 - (attempt - 1) * 0.30)
    if random.random() < fail_probability:
        raise RuntimeError(
            f"Attempt {attempt}: transient Azure storage timeout reading raw_cdr (simulated)"
        )
    raw_cdr = spark.read.table("trainer_demo.demo_11.raw_cdr")
    rejected = int(raw_cdr.count() * 0.03)   # ~3% rejection rate
    return {
        "attempt":        attempt,
        "raw_records":    raw_cdr.count(),
        "silver_records": raw_cdr.count() - rejected,
        "rejected":       rejected,
    }

MAX_RETRIES      = 3
RETRY_INTERVAL_S = 3   # shortened for demo (real config: 60 000 ms)

print(f"cleanse_cdr task starting (max_retries={MAX_RETRIES}, interval={RETRY_INTERVAL_S}s demo / 60s real)")
print(f"{'='*70}\n")

random.seed(42)   # reproducible output for the demo
result = None
for attempt in range(1, MAX_RETRIES + 2):
    print(f"  Attempt {attempt}/{MAX_RETRIES + 1}: ", end="", flush=True)
    try:
        result = run_cleanse_cdr(attempt)
        print(f"SUCCESS  ->  {result['silver_records']:,} silver records written")
        print(f"\n  Task completed on attempt {attempt} of {MAX_RETRIES + 1}.")
        break
    except RuntimeError as e:
        if attempt <= MAX_RETRIES:
            print(f"FAILED  ->  {e}")
            print(f"            Databricks waits {RETRY_INTERVAL_S}s before retry ...")
            time.sleep(RETRY_INTERVAL_S)
        else:
            print(f"FAILED  ->  All {MAX_RETRIES} retries exhausted.")
            print(f"            Task status set to FAILED. error_handler task will run next.")

if result:
    print(f"\n  Summary: {result['raw_records']:,} raw -> {result['silver_records']:,} silver ({result['rejected']:,} rejected)")


In [ ]:
# -- Exponential backoff for continuous jobs (fraud detection scenario) -------
# Trainer: visualise the exponential backoff that Databricks applies when a
# continuous job (e.g. NexaTel real-time fraud detection) fails repeatedly.
# This illustrates the algorithm described in the learn unit.

print("=== Exponential Backoff - NexaTel Fraud Detection (continuous job) ===\n")
print("Scenario: the real-time CDR fraud detection job fails 8 times in a row")
print("due to a degraded network path to the CDR source table.\n")

# Simulate Databricks exponential backoff (illustrative approximation)
BASE_DELAY_S  = 10     # starting delay
MAX_DELAY_S   = 3600   # 1-hour cap (system maximum)

print(f"  {'Run':<5} {'Result':<10} {'Consec Fails':<14} {'Wait before next run (s)':<26} {'Wait (min)'}")
print(f"  {'-'*5} {'-'*10} {'-'*14} {'-'*26} {'-'*10}")

consecutive_failures = 0
for run in range(1, 18):
    # Simulate: runs 1-8 fail, 9+ succeed
    is_failure = run <= 8

    if is_failure:
        consecutive_failures += 1
        wait_s = min(BASE_DELAY_S * (2 ** (consecutive_failures - 1)), MAX_DELAY_S)
        status = "FAILED"
        note = ""
    else:
        consecutive_failures = 0
        wait_s = BASE_DELAY_S  # resets after successful run
        status = "SUCCESS"
        note = " <- backoff resets" if run == 9 else ""

    print(f"  {run:<5} {status:<10} {consecutive_failures:<14} {wait_s:<26} {wait_s/60:<10.1f}{note}")

total_downtime = sum(
    min(BASE_DELAY_S * (2 ** i), MAX_DELAY_S)
    for i in range(8)
)
print(f"\n  Total time in backoff (runs 1-8): {total_downtime}s = {total_downtime/60:.1f} minutes")

print("""
Key points for the audience:
  1. Continuous jobs restart automatically - no manual intervention needed.
  2. Exponential backoff prevents hammering a degraded resource with rapid retries.
  3. After a successful run, the backoff sequence RESETS (back to BASE_DELAY_S).
  4. The 'Job details' panel shows: consecutive failures, time to next retry.
  5. Combine with a Duration warning notification to alert when downtime exceeds SLA.
  6. For non-continuous jobs: configure task-level retry policies (separate UI setting).
""")
